# Lab 1 — Text-to-SQL Pipeline (Solutions)

In this lab you build the core of a "talk to your data" system: a pipeline that
takes a natural-language question, produces a SQL query, validates it, executes it,
and judges the answer.

## Learning objectives

By the end of this lab you will be able to:

1. Turn a database schema into a useful **system prompt** for a code-generating LLM.
2. Use **few-shot examples** to steer query style and coverage.
3. Apply **static checks** to a generated SQL string using an AST parser (`sqlglot`),
   rejecting non-`SELECT` statements, unknown tables, and unbounded scans.
4. Execute the query safely against SQLite and return a tidy result.
5. Evaluate correctness two ways: **dynamic comparison** to a gold query when one exists,
   and **LLM-as-judge** when it does not.
6. Assemble all of the above into a small **evaluation harness** that you can run over a
   test set.

## Prerequisites

- `python seed_db.py` has been run and `shop.db` exists next to this notebook.
- `OPENAI_API_KEY` is set in your environment or in a `.env` file in this folder.

## 1. Setup

In [ ]:
# If you have not yet installed dependencies:
# %pip install -r requirements.txt

In [ ]:
import os
import json
import sqlite3
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
import sqlglot
from sqlglot import exp

load_dotenv()

DB_PATH = Path("shop.db")
assert DB_PATH.exists(), "Run `python seed_db.py` first to create shop.db."

MODEL = "gpt-4o-mini"     # cheap & fast; swap for gpt-4o for higher quality
JUDGE_MODEL = "gpt-4o-mini"

client = OpenAI()  # reads OPENAI_API_KEY from the environment

## 2. The database

`shop.db` models a tiny e-commerce backend. Five tables:

| table        | purpose                                              |
|--------------|------------------------------------------------------|
| `customers`  | One row per customer (name, email, country).         |
| `products`   | Catalogue (name, category, price, stock).            |
| `orders`     | Header per order (customer, date, status, total).    |
| `order_items`| Line items: which products were in which order.      |
| `reviews`    | Star ratings + free-text comments.                   |

We'll work directly with `sqlite3` to keep dependencies minimal.

In [ ]:
def query_df(sql: str, params: tuple = ()) -> pd.DataFrame:
    """Run SQL against shop.db and return the result as a DataFrame."""
    with sqlite3.connect(DB_PATH) as conn:
        return pd.read_sql_query(sql, conn, params=params)

query_df("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;")

In [ ]:
# Peek at a few rows from each table
for table in ["customers", "products", "orders", "order_items", "reviews"]:
    print(f"--- {table} ---")
    print(query_df(f"SELECT * FROM {table} LIMIT 3").to_string(index=False))
    print()

## 3. Naive text-to-SQL

The simplest thing that could work: tell the model "you are a SQL expert", give it the
question, and parse out a SQL string. No schema, no examples, no checks.

Let's see what goes wrong.

In [ ]:
def naive_translate(question: str) -> str:
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are an expert SQL author. Reply with a single SQL query and no commentary."},
            {"role": "user", "content": question},
        ],
        temperature=0,
    )
    return resp.choices[0].message.content.strip()

print(naive_translate("How many customers do we have?"))

Try a more ambiguous question. Without the schema the model has to **guess** column names —
sometimes it guesses right, sometimes it invents fields that don't exist.

In [ ]:
print(naive_translate("Which products have we sold most of, by quantity?"))

## 4. Schema-aware prompting

Giving the model the schema is the single biggest quality improvement. We'll extract the
`CREATE TABLE` statements from SQLite's catalogue and inject them into the system prompt.

### Why DDL specifically?

`CREATE TABLE` statements carry column names, types, NOT NULL constraints, CHECK
constraints, and FK references — everything the model needs to write a valid query.
Sending raw rows is wasteful and risks leaking data into prompts; sending DDL is compact
and lossless.

In [ ]:
def get_schema_ddl() -> str:
    sql = """
        SELECT sql FROM sqlite_master
        WHERE type IN ('table','view') AND name NOT LIKE 'sqlite_%'
        ORDER BY name;
    """
    with sqlite3.connect(DB_PATH) as conn:
        rows = conn.execute(sql).fetchall()
    return "\n\n".join(r[0] for r in rows if r[0])

print(get_schema_ddl())

### TODO 4.1 — implement `translate_to_sql`

Write a function that takes a natural-language `question` and returns a SQL string.

Requirements:

- Include the schema DDL in the system prompt.
- Instruct the model to return **only** the SQL — no markdown fences, no commentary.
- Strip any code-fence wrapping (` ```sql ... ``` `) defensively, in case the model adds it
  anyway.
- Use `temperature=0` for reproducibility.

In [ ]:
def _strip_fence(text: str) -> str:
    t = text.strip()
    if t.startswith("```"):
        # Drop the opening fence (with optional language tag) and the closing fence.
        t = t.split("\n", 1)[1] if "\n" in t else t
        if t.endswith("```"):
            t = t[: -3]
    return t.strip()


def translate_to_sql(question: str) -> str:
    system = (
        "You are an expert SQLite SQL author.\n"
        "Given a user question, return ONE SQL query that answers it.\n"
        "Use only the tables and columns shown in the schema below.\n"
        "Reply with the SQL only — no prose, no markdown fences.\n\n"
        "Schema:\n" + get_schema_ddl()
    )
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": question},
        ],
        temperature=0,
    )
    return _strip_fence(resp.choices[0].message.content)


print(translate_to_sql("How many orders are in 'delivered' status?"))

In [ ]:
# Try it on a few realistic questions
questions = [
    "How many customers are from Hungary (country code HU)?",
    "What is the average rating per product category?",
    "Which customer has spent the most overall?",
]
for q in questions:
    print(f"Q: {q}")
    print(translate_to_sql(q))
    print()

## 5. Few-shot examples

Schema alone gets us most of the way. Few-shot examples close the gap on style and edge
cases — `JOIN` ordering, `GROUP BY` discipline, how to spell "this year", etc.

Below we add three examples to the prompt. Notice that they share *style* with what we
want back (lowercase keywords, explicit aliases, no `SELECT *`).

In [ ]:
FEW_SHOT = [
    {
        "q": "How many customers signed up in 2024?",
        "sql": "select count(*) as n_customers from customers where strftime('%Y', created_at) = '2024';",
    },
    {
        "q": "Top 5 products by total revenue.",
        "sql": (
            "select p.name, sum(oi.quantity * oi.unit_price) as revenue "
            "from order_items oi join products p on p.product_id = oi.product_id "
            "group by p.name order by revenue desc limit 5;"
        ),
    },
    {
        "q": "Average rating for product 'Wireless Mouse'.",
        "sql": (
            "select avg(r.rating) as avg_rating from reviews r "
            "join products p on p.product_id = r.product_id where p.name = 'Wireless Mouse';"
        ),
    },
]


def translate_to_sql_fewshot(question: str) -> str:
    system = (
        "You are an expert SQLite SQL author.\n"
        "Return ONE SQL query that answers the user's question.\n"
        "Use only the tables and columns shown in the schema. SQL only — no prose, no fences.\n\n"
        "Schema:\n" + get_schema_ddl()
    )
    messages = [{"role": "system", "content": system}]
    for ex in FEW_SHOT:
        messages.append({"role": "user", "content": ex["q"]})
        messages.append({"role": "assistant", "content": ex["sql"]})
    messages.append({"role": "user", "content": question})

    resp = client.chat.completions.create(model=MODEL, messages=messages, temperature=0)
    return resp.choices[0].message.content.strip()


print(translate_to_sql_fewshot("Revenue by product category, descending."))

## 6. Static checks

Never run a model-generated query without first asking: *is it well-formed, is it safe,
does it only touch tables we expect?* This is a cheap, deterministic line of defence
that doesn't need to call the LLM again.

We'll use [`sqlglot`](https://github.com/tobymao/sqlglot), a pure-Python SQL parser, to
turn the query into an AST and inspect it.

Three checks we care about right now (the security lab adds more):

1. **Parses** — `sqlglot.parse_one(sql, dialect='sqlite')` doesn't raise.
2. **SELECT-only** — the top-level node is a `Select`, not an `Insert`/`Update`/`Delete`/`Drop`.
3. **Tables are in the allow-list** — every referenced table belongs to our schema.

### TODO 6.1 — `is_select_only`

In [ ]:
FORBIDDEN = (
    exp.Insert, exp.Update, exp.Delete, exp.Drop, exp.Create,
    exp.AlterTable, exp.TruncateTable,
)


def is_select_only(sql: str) -> bool:
    try:
        tree = sqlglot.parse_one(sql, dialect="sqlite")
    except Exception:
        return False
    if not isinstance(tree, exp.Select):
        return False
    # Defence in depth: reject any forbidden node anywhere in the tree (e.g. CTEs
    # that smuggle in a DELETE on a dialect that allows it).
    return not any(isinstance(node, FORBIDDEN) for node in tree.walk())


assert is_select_only("SELECT 1") is True
assert is_select_only("select * from customers where customer_id = 1") is True
assert is_select_only("DELETE FROM customers") is False
assert is_select_only("DROP TABLE customers") is False
assert is_select_only("UPDATE customers SET email = 'x'") is False
print("ok")

### TODO 6.2 — `referenced_tables`

In [ ]:
def referenced_tables(sql: str) -> set[str]:
    tree = sqlglot.parse_one(sql, dialect="sqlite")
    return {t.name for t in tree.find_all(exp.Table)}


assert referenced_tables("select * from customers c join orders o on o.customer_id = c.customer_id") == {"customers", "orders"}
print("ok")

### TODO 6.3 — `validate_sql`

Combine the checks. Return `(ok, reason)`. `ok` is `True` only when:

- the query parses,
- it is SELECT-only,
- every referenced table is in `allowed_tables`.

In [ ]:
ALLOWED_TABLES = {"customers", "products", "orders", "order_items", "reviews"}


def validate_sql(sql: str, allowed: set[str] = ALLOWED_TABLES) -> tuple[bool, str]:
    try:
        sqlglot.parse_one(sql, dialect="sqlite")
    except Exception as e:
        return False, f"parse_error: {e}"
    if not is_select_only(sql):
        return False, "not_select_only"
    bad = referenced_tables(sql) - allowed
    if bad:
        return False, f"unknown_tables: {sorted(bad)}"
    return True, "ok"


print(validate_sql("SELECT name FROM customers LIMIT 5"))
print(validate_sql("DELETE FROM customers"))
print(validate_sql("SELECT * FROM secret_table"))

## 7. Execution

Static checks passed — now we run the query. Two safety guards still apply at run time:

- **Read-only connection** (`mode=ro` via SQLite URI).
- **Row cap** — slice the DataFrame to a maximum number of rows after fetch.

In [ ]:
MAX_ROWS = 100


def execute_sql(sql: str, max_rows: int = MAX_ROWS) -> pd.DataFrame:
    uri = f"file:{DB_PATH.as_posix()}?mode=ro"
    with sqlite3.connect(uri, uri=True) as conn:
        df = pd.read_sql_query(sql, conn)
    return df.head(max_rows)


sql = translate_to_sql_fewshot("Top 3 customers by number of orders.")
print("Generated SQL:\n", sql, "\n")
ok, why = validate_sql(sql)
print("Validation:", ok, why)
if ok:
    display = execute_sql(sql)
    print(display.to_string(index=False))

## 8. Dynamic correctness (gold answer)

When we have a **gold SQL** for a question, we can run both queries and compare result
sets. This is the cheapest, most reliable signal in a regression suite: if the column
sets and rows match, the candidate is correct *for this question* regardless of how the
SQL is shaped.

### TODO 8.1 — `results_match`

Implement a comparison that is robust to:

- **Row order** (most analytics questions don't specify an order)
- **Column order** (different SQL can return columns in different orders)
- **Column naming** when the values are identical but the alias differs — see hint

In [ ]:
from collections import Counter


def results_match(a: pd.DataFrame, b: pd.DataFrame, ignore_column_names: bool = False) -> bool:
    if ignore_column_names:
        return Counter(map(tuple, a.values.tolist())) == Counter(map(tuple, b.values.tolist()))
    if set(a.columns) != set(b.columns):
        return False
    b_aligned = b[list(a.columns)]
    return Counter(map(tuple, a.values.tolist())) == Counter(map(tuple, b_aligned.values.tolist()))


a = pd.DataFrame({"x": [1, 2, 3]})
b = pd.DataFrame({"x": [3, 1, 2]})
assert results_match(a, b)
c = pd.DataFrame({"y": [3, 1, 2]})
assert not results_match(a, c)
assert results_match(a, c, ignore_column_names=True)
print("ok")

In [ ]:
# Compare a generated query against a gold one
question = "How many delivered orders are there?"
gold = "SELECT COUNT(*) AS n FROM orders WHERE status = 'delivered'"

candidate = translate_to_sql_fewshot(question)
print("Candidate:", candidate)

df_gold = execute_sql(gold)
df_cand = execute_sql(candidate)
print("\nGold:\n", df_gold)
print("\nCandidate:\n", df_cand)
print("\nMatch:", results_match(df_gold, df_cand, ignore_column_names=True))

## 9. LLM-as-judge (no gold answer)

Gold queries are expensive to write and cover only a tiny fraction of real questions. For
the long tail, we ask a model to judge whether the SQL and its result *plausibly answer*
the user's question.

The judge prompt has three jobs:

1. **Constrain the output** to a small JSON schema so it is machine-parseable.
2. **Force a verdict** (`pass` / `fail`) and a short reason.
3. **Pass a result preview**, not the whole result, to keep prompts cheap and to avoid
   biasing the judge with sheer volume of data.

Limitations to know:

- A judge is correlated with the generator (same model family, similar blind spots).
  Use a *different* model when the budget allows.
- Judges drift. Pin the prompt and re-evaluate on a held-out gold set periodically.

### TODO 9.1 — `judge_answer`

In [ ]:
JUDGE_SYSTEM = """You judge whether a SQL query plus its result answers a user's question.
Reply with a single JSON object: {"verdict": "pass" | "fail", "reason": "<one short sentence>"}.
Do not include any other text."""


def judge_answer(question: str, sql: str, result_df: pd.DataFrame) -> dict:
    preview = result_df.head(10).to_csv(index=False)
    user_msg = (
        f"Question:\n{question}\n\n"
        f"SQL:\n{sql}\n\n"
        f"Result (first 10 rows, CSV):\n{preview}"
    )
    resp = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM},
            {"role": "user", "content": user_msg},
        ],
        temperature=0,
        response_format={"type": "json_object"},
    )
    return json.loads(resp.choices[0].message.content)


verdict = judge_answer(
    "How many delivered orders are there?",
    "SELECT COUNT(*) AS n FROM orders WHERE status = 'delivered'",
    execute_sql("SELECT COUNT(*) AS n FROM orders WHERE status = 'delivered'"),
)
print(verdict)

## 10. Putting it together — a small eval harness

We now have all the pieces. Let's run them as a pipeline over a test set with mixed
cases — some with a gold SQL (so we use the dynamic check), some without (so we fall
back to the LLM judge).

In [ ]:
TEST_SET = [
    {
        "question": "How many customers are from Hungary (country code HU)?",
        "gold_sql": "SELECT COUNT(*) AS n FROM customers WHERE country = 'HU'",
    },
    {
        "question": "How many orders have status 'cancelled'?",
        "gold_sql": "SELECT COUNT(*) AS n FROM orders WHERE status = 'cancelled'",
    },
    {
        "question": "Top 3 product categories by revenue.",
        "gold_sql": None,   # no gold — fall back to LLM judge
    },
    {
        "question": "Which customer wrote the most reviews?",
        "gold_sql": None,
    },
    {
        "question": "Delete all customers from Germany.",
        "gold_sql": None,   # adversarial — should be rejected by validate_sql
    },
]

### TODO 10.1 — `evaluate_pipeline`

In [ ]:
def evaluate_pipeline(test_set: list[dict]) -> pd.DataFrame:
    rows = []
    for item in test_set:
        q = item["question"]
        gold = item.get("gold_sql")
        sql = translate_to_sql_fewshot(q)
        ok, why = validate_sql(sql)
        if not ok:
            rows.append({"question": q, "sql": sql, "valid": False, "status": "rejected", "reason": why})
            continue
        try:
            df = execute_sql(sql)
        except Exception as e:
            rows.append({"question": q, "sql": sql, "valid": True, "status": "error", "reason": str(e)})
            continue
        if gold:
            df_gold = execute_sql(gold)
            passed = results_match(df_gold, df, ignore_column_names=True)
            rows.append({
                "question": q, "sql": sql, "valid": True,
                "status": "pass" if passed else "fail",
                "reason": "match_gold" if passed else "mismatch_gold",
            })
        else:
            verdict = judge_answer(q, sql, df)
            rows.append({
                "question": q, "sql": sql, "valid": True,
                "status": verdict.get("verdict", "fail"),
                "reason": verdict.get("reason", ""),
            })
    return pd.DataFrame(rows)


report = evaluate_pipeline(TEST_SET)
with pd.option_context("display.max_colwidth", 80):
    print(report.to_string(index=False))

## Summary

You now have:

- A schema-aware, few-shot text-to-SQL translator.
- A static validator that rejects non-`SELECT` and out-of-schema queries before they
  touch the database.
- A read-only execution path with a row cap.
- Two complementary correctness signals: result-set comparison (when a gold SQL exists)
  and LLM-as-judge (when it doesn't).
- A harness that ties it all together into a measurable, repeatable evaluation.

In Lab 2 we treat the same system from a **security** angle: what happens when the user
is hostile, or when one user's data must be invisible to another?

## Exercises

1. **Error recovery.** When `execute_sql` raises (e.g. an ambiguous column), feed the
   error message back to the translator and let it try again. Cap retries at 2.
2. **Self-consistency.** Generate `n=3` candidate SQLs with `temperature=0.7`, execute
   each, and return the result that the majority agree on.
3. **Cross-judge.** Replace `judge_answer` with two judges from different model families
   and only mark `pass` when both agree.
4. **Cost report.** Extend `evaluate_pipeline` to record `usage.total_tokens` per call
   and report average tokens per question.